# 05 · Video pipeline design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb)

*Part III · group · 15 min*

> 🇪🇸 **Diseño de un pipeline de vídeo** — Convertir un archivo de vídeo real en un tensor, observar qué se pierde al muestrear fotogramas y diseñar formas tensoriales que respeten el significado del tiempo.

Process one pinned real video end to end, then use what its axes actually mean to design two downstream video pipelines.

## What you will be able to do

- Decode a pinned real video file into an order-4 tensor and name every axis.
- Measure how much temporal information a frame-sampling decision keeps and discards.
- Design tensor shapes for video-level and timestep-level prediction systems.
- Explain the memory trade-off between padding variable-length clips and fixed-frame sampling.

> 🇪🇸 **Lo que podrás hacer:**

> - Decodificar un archivo de vídeo real y verificado en un tensor de orden 4, nombrando cada eje.
> - Medir cuánta información temporal conserva y descarta una decisión de muestreo de fotogramas.
> - Diseñar formas tensoriales para sistemas de predicción a nivel de vídeo y a nivel de timestep.
> - Explicar el compromiso de memoria entre rellenar clips de longitud variable y muestrear un número fijo de fotogramas.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
%pip install -q "imageio[ffmpeg]"

import hashlib
import io
import urllib.request

import numpy as np
import imageio.v3 as iio
import matplotlib.pyplot as plt

# Real clip: "Tormenta en l'Almadrava" by Nicolas Vigier, CC0.
# https://commons.wikimedia.org/wiki/File:Tormenta_en_l%27Almadrava.webm
VIDEO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)
VIDEO_SHA256 = "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"


def fetch_verified_video(url, expected_sha256, n_frames=16, stride=45):
    # Verify the real file, decode the whole stream, retain only sparse frames.
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()

    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch: expected {expected_sha256}, got {got}"
        )

    kept_frames = []
    kept_source_indices = []
    total_frames = 0

    for i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        total_frames = i + 1
        if i % stride == 0 and len(kept_frames) < n_frames:
            kept_frames.append(frame)
            kept_source_indices.append(i)

    clip = np.stack(kept_frames)
    return clip, np.asarray(kept_source_indices), total_frames


clip, kept_source_indices, total_frames = fetch_verified_video(
    VIDEO_URL, VIDEO_SHA256
)

assert clip.shape == (16, 540, 960, 3), f"unexpected clip shape {clip.shape}"
assert total_frames == 720, f"unexpected frame count {total_frames}"

print("retained tensor:", clip.shape, clip.dtype)
print("source frames:", total_frames)
print("source indices retained:", kept_source_indices.tolist())
print("RAM retained:", f"{clip.nbytes / 1024**2:.1f} MB")

## Why this matters

A video model never receives “a video” in the abstract. A real file is decoded into axes, and every preprocessing choice decides what information survives.

Here the camera recorded **720 frames**. The pipeline keeps only **16** of them in the in-memory tensor `(16, 540, 960, 3)`. That is efficient, but it also means most temporal measurements are deliberately discarded.

This is the concept for the whole section:

> **A video pipeline is a sequence of decisions about which axes survive, which axes move, and which information is discarded.**

> 🇪🇸 **Por qué importa:** un modelo no recibe “un vídeo” de forma abstracta. El archivo real se decodifica en ejes y cada decisión de preprocesamiento determina qué información sobrevive. Aquí la cámara grabó 720 fotogramas, pero el tensor en memoria conserva solo 16. La eficiencia tiene un costo: se descarta información temporal real.

### Predict → Run → Explain

Before each exercise, predict both the **shape** and the **meaning of every axis**. After running code, explain what was kept and what was lost.

> 🇪🇸 **Predice → Ejecuta → Explica:** antes de cada ejercicio, predice tanto la forma como el significado de cada eje. Después de ejecutar, explica qué información se conservó y cuál se perdió.


## Exercise 1 — watch a real video become a tensor

Start with the measured data, not a diagram. `clip` contains 16 real frames sampled from the verified 720-frame source video.

**Predict first:** what does each axis in `(16, 540, 960, 3)` count? What percentage of the recorded frames did this pipeline retain?

> 🇪🇸 Empieza con datos medidos, no con un diagrama. `clip` contiene 16 fotogramas reales muestreados del vídeo verificado de 720 fotogramas. **Predice primero:** ¿qué cuenta cada eje y qué porcentaje de los fotogramas grabados conservó el pipeline?


In [ ]:
# TODO 1: Print clip.shape and clip.dtype.
# Name the meaning of axes 0, 1, 2 and 3.
#
# TODO 2: Using len(clip) and total_frames, compute:
#   - fraction of recorded frames retained
#   - fraction discarded
#
# TODO 3: Inspect kept_source_indices.
# Explain why clip[1] is NOT source frame 1.


In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print("shape:", clip.shape, "dtype:", clip.dtype)
print("axes: (retained time, height, width, colour)")

retained_fraction = len(clip) / total_frames
discarded_fraction = 1 - retained_fraction

print("retained:", f"{retained_fraction:.2%}")
print("discarded:", f"{discarded_fraction:.2%}")
print("clip[1] came from source frame", kept_source_indices[1])

# Screenshot-friendly visual computed from the real file.
fig, axes = plt.subplots(2, 1, figsize=(11, 6))

axes[0].scatter(
    np.arange(total_frames), np.zeros(total_frames),
    s=7, alpha=0.18, label="recorded frame"
)
axes[0].scatter(
    kept_source_indices, np.zeros_like(kept_source_indices),
    s=45, label="retained in tensor"
)
axes[0].set_yticks([])
axes[0].set_xlim(-5, total_frames + 5)
axes[0].set_xlabel("source frame index")
axes[0].set_title(
    f"Real video sampling: {len(clip)} of {total_frames} frames retained "
    f"({retained_fraction:.1%})"
)
axes[0].legend(loc="upper right")

preview_slots = [0, 5, 10, 15]
strip = np.concatenate([clip[k] for k in preview_slots], axis=1)
axes[1].imshow(strip)
axes[1].set_title(
    "Four retained real frames — source indices "
    + ", ".join(str(kept_source_indices[k]) for k in preview_slots)
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Interactive frame browser in Colab.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
from IPython.display import display

def show_retained_frame(k):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(clip[k])
    ax.set_title(
        f"clip[{k}] = source frame {kept_source_indices[k]} of {total_frames}"
    )
    ax.axis("off")
    plt.show()

slider = widgets.IntSlider(
    value=0, min=0, max=len(clip)-1, step=1,
    description="retained frame", continuous_update=False
)
display(widgets.interactive(show_retained_frame, k=slider))


## Exercise 2 — one real input tensor, two different systems

Now use the real decoded-video convention `(T,H,W,C)` as the anchor and design two downstream systems:

- **Tech:** a short-video recommender that returns one embedding per video.
- **Biotech:** a surgical-video model that returns one phase-label distribution per timestep.

These are **design scenarios**, not additional datasets. Their output shapes are architectural choices; the input-axis reasoning is grounded in the real decoded tensor above.

> 🇪🇸 Usa la convención real `(T,H,W,C)` como punto de partida para diseñar dos sistemas. Son **escenarios de diseño**, no conjuntos de datos adicionales: las formas de salida son decisiones arquitectónicas, mientras que el razonamiento sobre los ejes parte del tensor real ya decodificado.


In [ ]:
# TODO 4: Fill in the shape at each stage for BOTH systems.
# Next to every shape, write what each axis counts.
#
# --- Tech: short-video recommender, one embedding per video -------------------
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...
#
# --- Biotech: surgical phase labelling, one label distribution per timestep --
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...
#
# Then answer: which system intentionally removes the time axis at the output?


In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
# One defensible design. Different sizes can also be correct if axis meanings
# and model goals are internally consistent.

# --- Tech: short-video recommender -------------------------------------------
# raw file          : bytes on disk, no tensor shape yet
# decoded frames    : (T, H, W, C)
# preprocessed batch: (32, 8, 224, 224, 3)   N, T, H, W, C
# model input       : (32, 8, 224, 224, 3)
# model output      : (32, 512)              N, embedding
# Time is intentionally collapsed into one vector per video.

# --- Biotech: surgical phase labelling ---------------------------------------
# raw file          : bytes on disk
# decoded frames    : (T, H, W, C)
# preprocessed batch: (4, 64, 224, 224, 3)   N, T, H, W, C
# model input       : (4, 64, 224, 224, 3)
# model output      : (4, 64, 12)            N, T, classes
# Time survives because the predicted phase can change by timestep.

print("real anchor shape:", clip.shape, "-> (T, H, W, C)")
print("tech output       :", (32, 512), "-> time collapsed")
print("biotech output    :", (4, 64, 12), "-> time preserved")


## Exercise 3 — ragged clips: pad or sample?

Real deployments receive videos with different durations. To make the memory cost visible without allocating an impossible image tensor, use a **stress-test design scenario** with clips lasting 30 s, 45 s, 2 min and 4 h at 30 fps.

The durations here are intentionally chosen design inputs, not measurements from the Wikimedia clip. The lesson is the tensor cost they imply.

> 🇪🇸 Los sistemas reales reciben vídeos con duraciones diferentes. Para visualizar el costo de memoria sin crear un tensor de imágenes imposible, usa un **escenario de estrés** con clips de 30 s, 45 s, 2 min y 4 h a 30 fps. Estas duraciones son entradas deliberadas del ejercicio, no mediciones del vídeo de Wikimedia.


In [ ]:
# TODO 5:
# Convert durations [30s, 45s, 2min, 4h] at 30 fps into frame counts.
# If all four are padded to the longest length, what is the boolean mask shape?
# What fraction of positions in that mask are invented padding?
#
# TODO 6:
# Compare that with sampling exactly 64 frames from every clip.
# What is gained? What real temporal information is lost?
#
# TODO 7:
# A surgical system adds 3 synchronized camera angles.
# Write one shape that keeps camera as its own axis and one that folds camera
# into the batch axis. When would keeping CAM explicit be necessary?


In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
durations_s = np.array([30, 45, 2 * 60, 4 * 60 * 60])
lengths = durations_s * 30
T_max = int(lengths.max())

mask = np.zeros((len(lengths), T_max), dtype=bool)
for i, n in enumerate(lengths):
    mask[i, :int(n)] = True

wasted = 1 - mask.sum() / mask.size

print("frame counts:", lengths.tolist())
print("mask shape:", mask.shape)
print("padding fraction:", f"{wasted:.2%}")

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.imshow(mask, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(4))
ax.set_yticklabels(["30 s", "45 s", "2 min", "4 h"])
ax.set_xlabel("frame index")
ax.set_title(
    f"Valid positions vs padding — {wasted:.2%} of the padded "
    "representation is invented"
)
plt.tight_layout()
plt.show()

sampled_batch_shape = (4, 64, 224, 224, 3)
print("fixed-sampling batch:", sampled_batch_shape)
# Gain: bounded, predictable memory.
# Cost: long clips are sampled more sparsely; temporal events can vanish.

explicit_camera = (4, 3, 64, 224, 224, 3)   # N, CAM, T, H, W, C
folded_camera = (12, 64, 224, 224, 3)       # N*CAM, T, H, W, C
print("camera explicit:", explicit_camera)
print("camera folded  :", folded_camera)
# Keep CAM explicit when the model must combine information across views.


## What just happened

You followed one real video from file bytes to a tensor and then used that concrete tensor to reason about larger systems.

1. The pinned WebM file decoded to **720 recorded frames**, while the in-memory tensor retained **16 real frames** with shape `(16, 540, 960, 3)`.
2. The real-data timeline made the sampling loss measurable: retaining 16 of 720 frames keeps only about **2.2%** of the recorded timesteps.
3. Two downstream systems can start from the same `(T,H,W,C)` convention and still need different outputs: a recommender may collapse time; timestep labelling must preserve it.
4. The ragged-length stress test showed why padding can be mathematically valid but operationally wasteful, while fixed sampling controls memory by discarding temporal information.
5. Camera, batch and time axes are not interchangeable just because they are dimensions of the same tensor.

> 🇪🇸 **Qué ocurrió:** seguiste un vídeo real desde los bytes del archivo hasta un tensor. El archivo contiene 720 fotogramas grabados, pero el tensor conserva 16 imágenes reales `(16,540,960,3)`, aproximadamente el 2.2% de los instantes registrados. Luego viste que conservar, colapsar, rellenar o muestrear un eje temporal son decisiones del pipeline con consecuencias distintas. La forma del tensor no es solo notación: registra qué información decidiste conservar.


---

## Done with this section

> 🇪🇸 **Fin de esta sección.**

Next up: **06 · Contraction with einsum** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)